In [ ]:
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch.optim as optim

# How to train extremely shallow network well?
# 
# I failed to align complex geometry with no-hidden layer.
# One of the Welch lab's video "Deep geometry ~" proposes this kind of challenge (https://github.com/WelchLabs/videos/blob/master/_2025/backprop_3/notebooks/Wide%20Training%20Example.ipynb)
# 
# Can I show UAT in real world? (with 100k neuron)

In [ ]:
im = Image.open("Baarle-Nassau_-_Baarle-Hertog-en no legend.png")
im_ndarray = np.array(im)[:,:,(1,2,3)]
plt.imshow(im_ndarray)

In [ ]:
overlay = im_ndarray.copy()
overlay[600,:]=np.array([0, 0, 0])
overlay[:,150]=np.array([0, 0, 0])
overlay[:,300]=np.array([0, 0, 0])
plt.imshow(overlay)
netherlands_color = im_ndarray[600, 150]
belgium_color = im_ndarray[600, 300]

In [ ]:
from scipy import signal

mask = ((im_ndarray-belgium_color)**2).sum(-1)==0

# netherlands_region=((im_ndarray-netherlands_color)**2).sum(-1)<500
# belgium_region=((im_ndarray-belgium_color)**2).sum(-1)<100

# fig, ax = plt.subplots(1, 2)
# ax[0].imshow(netherlands_region, cmap="gray")
# ax[1].imshow(belgium_region, cmap="gray")
kernel = np.array([
    [1, 1, 1],
    [1, 0, 1],
    [1, 1, 1],
])
p = mask.copy()

In [ ]:

# for i in range(5):
p = (signal.convolve2d(p, kernel, mode="same")>3)*1
plt.imshow(p, cmap="gray")

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, box
import osmnx as ox

# 1. 폴리곤 가져오기

lat, lon = 51.4440, 4.9320
lat = 51+26/60+31.2/3600
long = 4+55/60+59.9/3600
# 51°26'31.2"N 4°55'59.9"E
center = gpd.GeoSeries([Point(lon, lat)], crs=4326).to_crs(3857).iloc[0]

# 3️⃣ 반경 설정 (meters)
half_size = 2800  # 500m

minx = center.x - half_size
maxx = center.x + half_size
miny = center.y - half_size
maxy = center.y + half_size

bbox = box(minx, miny, maxx, maxy)
clipped = geom.intersection(bbox)

resolution = 5  # meters per pixel

width  = int((maxx - minx) / resolution)
height = int((maxy - miny) / resolution)

transform = Affine(resolution, 0, minx,
                   0, resolution, miny)

mask = rasterize(
    [(clipped, 1)],
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype=np.uint8
)

In [ ]:
plt.imshow(mask)

In [ ]:
mask[:,:,None].shape

In [ ]:
im = Image.fromarray(mask[:,:]*255, mode="L")
im.save("BN-border.png")

In [ ]:
device = "cpu"

In [ ]:
data = torch.from_numpy(np.array(Image.open("BN-border.png"), dtype=np.int64)).to(device)
h, w = data.shape
x = torch.arange(0, h)
y = torch.arange(0, w)
pixel_x, pixel_y = torch.meshgrid(x, y, indexing="ij")

x = torch.linspace(-1, 1, h, device=device)
y = torch.linspace(-1, 1, w, device=device)

grid_x, grid_y = torch.meshgrid(x, y, indexing="ij")
din = torch.stack([grid_x, grid_y], dim=-1).flatten(0, -2)
dout = data.flatten()
dout[dout>0]=1

In [ ]:
dout.sum()/(1120**2), din[0]

In [ ]:
torch.manual_seed(42)

from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(din, dout)
dataloader = DataLoader(dataset, batch_size=1000, shuffle=True)
test_dataloader = DataLoader(dataset, batch_size=1000, shuffle=False)

In [ ]:
len(dataset)

In [ ]:
@torch.no_grad
def viz(model:nn.Module):
    model.eval()
    map = torch.from_numpy(np.array(Image.open("BN-border.png"), dtype=np.float32))
    probs = []
    for batch_X, _ in test_dataloader:
        probs.append(nn.functional.softmax(model(batch_X), -1))
    probs = torch.cat(probs, 0)
    boundary = (probs[:, 1]>0.5).reshape(h, w)
    map = np.where(boundary.cpu(), torch.ones_like(map)*0.5, map)
    model.train()
    return map

In [ ]:
# 32 wide
net = nn.Sequential(*[
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
]).to(device)

optimizer = torch.optim.Adam(net.parameters(), lr=0.005)

In [ ]:

num_epochs = 10
for epoch in range(num_epochs):
    epoch_loss = 0.0
    num_batches = 0
    
    for batch_X, batch_y in dataloader:
        outputs = net(batch_X)
        loss = nn.functional.cross_entropy(outputs, batch_y)

        # make w be independent!! -> more complex geometry?
        wl = net[0].weight
        # outer compute heavy than inner
        # wr[:,[0,1]] = wr[:,[1,0]]
        # wr[:, 1] *= -1
        # loss += torch.mm(wl, wr.transpose(0, 1)).mean()
        loss += (wl @ wl.transpose(0, 1)).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')
    plt.imshow(viz(net), cmap="gray", vmin=0, vmax=1)
    plt.show()

In [ ]:
plt.imshow(viz(net), cmap="gray", vmin=0, vmax=1)
plt.title("32 Wide with inner prod")
plt.axis("off")
plt.tight_layout()
plt.savefig("32wide.png", dpi=200)
plt.show()

In [ ]:
import tqdm

def train(net, optimizer=None, inner=False, num_epochs=10, eval_ratio=0.1, lr=0.005):
    if optimizer is None:
        optimizer = torch.optim.Adam(net.parameters(), lr=lr)

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        
        for batch_X, batch_y in tqdm.tqdm(dataloader):
            outputs = net(batch_X)
            loss = nn.functional.cross_entropy(outputs, batch_y)

            if inner:
                wl = net[0].weight
                loss += (wl @ wl.transpose(0, 1)).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
        
        avg_loss = epoch_loss / num_batches
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')
        if (epoch+1) % int(1/eval_ratio) == 0:
            plt.imshow(viz(net), cmap="gray", vmin=0, vmax=1)
            plt.show()

    return net

In [ ]:
# 32 wide
net = nn.Sequential(*[
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
]).to(device)

optimizer = torch.optim.Adam(net.parameters(), lr=0.005)

In [ ]:
train(net, optimizer, inner=True)

In [ ]:
net = nn.Sequential(*[
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
]).to(device)

In [ ]:
train(net, inner=True, num_epochs=30, eval_ratio=0.2)

In [ ]:
batch_X, batch_y = next(iter(dataloader))
cfg = {"cmap": "gray", "vmin": 0}

In [ ]:
# What should i gonna use bias to prevent harkey stick?
ineq_ratio = batch_y.sum()/len(batch_y)

v=torch.tensor([0., 1.], requires_grad=True)
opt = torch.optim.SGD([v], 1)
for i in range(50):
    (loss:=F.l1_loss(v.softmax(0)[0], ineq_ratio)).backward()
    opt.step()
    opt.zero_grad()
print(v, loss)
bias_term = v[1]-v[0] # softmax = softmax(a-c, b-c) -> translation
print(bias_term)

In [ ]:

# class Block(nn.Linear):
#     def forward(self, x):
#         y = F.linear(F.relu(x), self.weight, self.bias)
#         if 
#         return x + y
wide = 64
        
test_net = nn.Sequential(*[
    nn.Linear(2, wide),
    nn.Tanh(),
    nn.Linear(wide, wide),
    nn.Tanh(),
    nn.Linear(wide, 2),
]).to(device)

with torch.no_grad():
    rads = torch.linspace(0, 2*torch.pi, wide)
    polar = torch.stack([torch.cos(rads), torch.sin(rads)], dim=-1)
    test_net[0].weight = nn.Parameter(polar)
    test_net[0].bias = nn.Parameter(torch.zeros_like(test_net[0].bias))
    test_net[-1].weight = nn.Parameter(torch.ones_like(test_net[-1].weight)/wide)
    new_b1 = torch.zeros_like(test_net[-1].bias)
    new_b1[0] += bias_term
    test_net[-1].bias = nn.Parameter(new_b1)
    
optimizer = optim.Adam(test_net.parameters(), 0.01, betas=(0.95, 0.999))
losses = []
accs = []
size_t = len(batch_y)

updated = {}

for (n, p) in test_net.named_parameters():
    if "weight" in n:
        updated[n] = torch.zeros_like(p)

for _ in range(10000):
    pred = test_net(batch_X)
    (loss:=nn.functional.cross_entropy(pred, batch_y)).backward()
    # plt.imshow(pred.detach().reshape(50, 40), **cfg)
    # plt.colorbar()
    # plt.show()
    losses.append(loss.item())
    accs.append(((nn.functional.softmax(pred, -1)[:,1]>0.5)==batch_y).sum()/size_t)

    optimizer.step()

    for (n, p) in test_net.named_parameters():
        if n in updated and p.grad is not None:
            updated[n] += p.grad.abs()

    optimizer.zero_grad()

plt.plot(range(len(losses)), losses)
plt.plot(range(len(losses)), accs)
plt.ylim(0, 1)

In [ ]:
for (n, grad) in updated.items():
    print(f"{n}: {grad.norm()}")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 10))
    im1 = ax1.imshow(grad, **cfg)
    fig.colorbar(im1)
    im2 = ax2.imshow(test_net[int(n.split(".")[0])].weight.detach().abs(), **cfg)
    fig.colorbar(im2)
    plt.show()

In [ ]:
net = nn.Sequential(*[
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
]).to(device)

with torch.no_grad():
    rads = torch.linspace(0, 2*torch.pi, wide)
    polar = torch.stack([torch.cos(rads), torch.sin(rads)], dim=-1)
    net[0].weight = nn.Parameter(polar)
    net[0].bias = nn.Parameter(torch.zeros_like(net[0].bias))
    net[-1].weight = nn.Parameter(torch.ones_like(net[-1].weight)/wide)
    new_b1 = torch.zeros_like(net[-1].bias)
    new_b1[0] += bias_term
    net[-1].bias = nn.Parameter(new_b1)
train(net, num_epochs=30, eval_ratio=0.2)

In [ ]:
train(net, num_epochs=100, eval_ratio=0.2)

In [ ]:
wide = 128
net = nn.Sequential(*[
    nn.Linear(2, wide),
    nn.Tanh(),
    nn.Linear(wide, wide),
    nn.Tanh(),
    nn.Linear(wide, wide),
    nn.Tanh(),
    nn.Linear(wide, 2)
]).to(device)

with torch.no_grad():
    rads = torch.linspace(0, 2*torch.pi, wide)
    polar = torch.stack([torch.cos(rads), torch.sin(rads)], dim=-1)
    net[0].weight = nn.Parameter(polar)
    net[0].bias = nn.Parameter(torch.zeros_like(net[0].bias))
    net[-1].weight = nn.Parameter(torch.ones_like(net[-1].weight)/wide)
    new_b1 = torch.zeros_like(net[-1].bias)
    new_b1[0] += bias_term
    net[-1].bias = nn.Parameter(new_b1)
train(net, num_epochs=30, eval_ratio=0.2, lr=0.01)

In [ ]:
net = nn.Sequential(*[
    nn.Linear(2, 1024),
    nn.ReLU(),
    nn.Dropout(0.1),
    nn.Linear(1024, 2),
]).to(device)

train(net)

In [ ]:
net = nn.Sequential(*[
    nn.Linear(2, 1024),
    nn.LeakyReLU(),
    nn.Dropout(0.1),
    nn.Linear(1024, 2),
]).to(device)

train(net, inner=True)